In [ ]:
import os
import sys

if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

# ProcTHOR Franka Door + Pick-and-Place Demo

This notebook follows the original `main` notebook path: ProcTHOR house scene, real Franka insertion, and the policy-facing exo + wrist cameras.

Bootstrap reminders on Linux:
- `apt-get update && apt-get install -y libegl1 libgl1 libgles2 libglfw3 libosmesa6 libgl1-mesa-dri mesa-utils`
- `export MUJOCO_GL=egl`
- `export PYOPENGL_PLATFORM=egl`


In [ ]:
from pathlib import Path

import mujoco
import numpy as np
import pandas as pd
from PIL import Image
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
from IPython.display import Markdown, Video, display
from huggingface_hub import snapshot_download

from molmo_spaces.configs.robot_configs import FrankaRobotConfig
from molmo_spaces.molmo_spaces_constants import get_procthor_10k_houses, get_robot_path
from molmo_spaces.robots.franka import FrankaRobot
from molmo_spaces.robots.robot_views.franka_droid_view import FrankaDroidRobotView
from molmo_spaces.utils.lazy_loading_utils import install_scene_with_objects_and_grasps_from_path, install_uid
from olmo.eval.configure_real_robot import RealRobotVLAPolicy, RealRobotVLAPolicyConfig

In [ ]:
OUTPUT_DIR = Path('MolmoBot/demo_outputs/door_pick_place_shared_scene')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RENDER_HEIGHT = 360
RENDER_WIDTH = 640
POLICY_DT_MS = 66
ROBOT_CONFIG = FrankaRobotConfig(base_size=[0.5, 0.5, 0.75])
HOUSES = get_procthor_10k_houses(split='val')
HOUSE_XML_PATH = HOUSES['val'][0]['base']
SCENE_NAME = Path(HOUSE_XML_PATH).stem

DEFAULT_SCENE_CONFIG = {
    'robot_pos': [6.8, 9.75],
    'robot_yaw_deg': 90.0,
    'exo_pos': [0.1, 0.57, 0.66],
    'exo_quat': [-0.3633, -0.1241, 0.4263, 0.8191],
    'exo_fovy': 71.0,
}

TASK_SPECS = [
    {
        'task_id': 'door_open',
        'prompt': 'open the door',
        'duration_s': 18.0,
        'manual_note': 'review_pending',
        'scene_config': {
            'robot_pos': [8.62, 8.86],
            'robot_yaw_deg': 52.0,
            'exo_pos': [-0.05, 0.92, 0.86],
            'exo_quat': [-0.512, -0.214, 0.318, 0.771],
            'exo_fovy': 82.0,
        },
    },
    {
        'task_id': 'pick_and_place',
        'prompt': 'put the salt shaker in the bowl',
        'duration_s': 24.0,
        'manual_note': 'review_pending',
        'scene_config': {},
    },
]


class MockConfig:
    def __init__(self, policy_config):
        self.policy_config = policy_config


In [ ]:
def build_scene(task_spec=None):
    scene_config = dict(DEFAULT_SCENE_CONFIG)
    if task_spec is not None:
        scene_config.update(task_spec.get('scene_config', {}))

    install_scene_with_objects_and_grasps_from_path(HOUSE_XML_PATH)

    spec = mujoco.MjSpec.from_file(HOUSE_XML_PATH)
    robot_file_path = get_robot_path(ROBOT_CONFIG.name) / ROBOT_CONFIG.robot_xml_path
    robot_spec = mujoco.MjSpec.from_file(str(robot_file_path))

    FrankaRobot.add_robot_to_scene(
        ROBOT_CONFIG,
        spec,
        robot_spec,
        prefix=ROBOT_CONFIG.robot_namespace,
        pos=scene_config['robot_pos'],
        quat=R.from_euler('z', scene_config['robot_yaw_deg'], degrees=True).as_quat(scalar_first=True),
    )

    spec.camera(ROBOT_CONFIG.robot_namespace + 'gripper/wrist_camera').resolution = [RENDER_WIDTH, RENDER_HEIGHT]
    spec.body(ROBOT_CONFIG.robot_namespace + 'fr3_link0').add_camera(
        pos=scene_config['exo_pos'],
        quat=scene_config['exo_quat'],
        fovy=scene_config['exo_fovy'],
        resolution=[RENDER_WIDTH, RENDER_HEIGHT],
        name='robot_0/exo_camera_1',
    )

    bowl_xml_path = install_uid('Bowl_3')
    bowl_spec = mujoco.MjSpec.from_file(str(bowl_xml_path))
    root_body = bowl_spec.worldbody.first_body()
    receptacle_frame = spec.worldbody.add_frame(
        pos=[7.1, 10.2, 1.01],
        quat=R.from_euler('x', 90, degrees=True).as_quat(scalar_first=True),
    )
    receptacle_frame.attach_body(root_body, prefix='place_receptacle/')

    model = spec.compile()
    data = mujoco.MjData(model)
    view = FrankaDroidRobotView(data, ROBOT_CONFIG.robot_namespace)
    view.set_qpos_dict(ROBOT_CONFIG.init_qpos)
    mujoco.mj_forward(model, data)
    for mg_id in view.move_group_ids():
        mg = view.get_move_group(mg_id)
        mg.ctrl = mg.noop_ctrl
    mujoco.mj_forward(model, data)

    renderer = mujoco.Renderer(model, RENDER_HEIGHT, RENDER_WIDTH)
    scene_option = mujoco.MjvOption()
    scene_option.sitegroup = 0
    return {
        'model': model,
        'data': data,
        'view': view,
        'renderer': renderer,
        'scene_option': scene_option,
        'scene_config': scene_config,
    }


def render_obs(scene_ctx):
    renderer = scene_ctx['renderer']
    data = scene_ctx['data']
    scene_option = scene_ctx['scene_option']
    renderer.update_scene(data, camera='robot_0/exo_camera_1', scene_option=scene_option)
    exo_img = renderer.render()
    renderer.update_scene(data, camera='robot_0/gripper/wrist_camera', scene_option=scene_option)
    wrist_img = renderer.render()
    return {
        'exo_camera_1': exo_img,
        'wrist_camera': wrist_img,
    }


def stacked_frame(obs):
    return np.hstack([obs['exo_camera_1'], obs['wrist_camera']])


In [ ]:
scene_ctx = build_scene(TASK_SPECS[0])
preview = stacked_frame(render_obs(scene_ctx))
scene_ctx['renderer'].close()
Image.fromarray(preview)


In [ ]:
ckpt_path = snapshot_download('allenai/MolmoBot-DROID')

policy_config = RealRobotVLAPolicyConfig()
policy_config.checkpoint_path = ckpt_path
policy_config.action_type = 'joint_pos'
policy_config.action_keys['arm'] = 'joint_pos'

policy = RealRobotVLAPolicy(config=MockConfig(policy_config), task_type='manipulation')

In [ ]:
def run_rollout(task_spec, output_dir=OUTPUT_DIR, policy_dt_ms=POLICY_DT_MS):
    scene_ctx = build_scene(task_spec)
    model = scene_ctx['model']
    data = scene_ctx['data']
    view = scene_ctx['view']

    frames = []
    num_steps = round(task_spec['duration_s'] * 1000 / policy_dt_ms)
    for _ in tqdm(range(num_steps), desc=task_spec['task_id']):
        obs = render_obs(scene_ctx)
        jp = view.get_move_group('arm').joint_pos
        gripper_input = view.get_move_group('gripper').joint_pos
        policy_obs = {
            'task': task_spec['prompt'],
            'qpos': {
                'arm': jp,
                'gripper': gripper_input,
            },
            **obs,
        }
        frames.append(stacked_frame(obs))

        action = policy.get_action(policy_obs)
        for mg_id, ctrl in action.items():
            view.get_move_group(mg_id).ctrl = ctrl

        nstep = max(1, policy_dt_ms // max(1, round(model.opt.timestep * 1000)))
        mujoco.mj_step(model, data, nstep=nstep)

    video_path = output_dir / f"{task_spec['task_id']}.mp4"
    ImageSequenceClip(frames, fps=round(1000 / policy_dt_ms)).write_videofile(
        str(video_path),
        codec='libx264',
        audio=False,
        logger=None,
    )
    scene_ctx['renderer'].close()
    return {
        'task_id': task_spec['task_id'],
        'prompt': task_spec['prompt'],
        'scene': SCENE_NAME,
        'video_path': str(video_path),
        'manual_note': task_spec['manual_note'],
    }


def run_task_suite(task_specs):
    return pd.DataFrame([run_rollout(task_spec) for task_spec in task_specs])


In [ ]:
results_df = run_task_suite(TASK_SPECS)
results_df

In [ ]:
for record in results_df.to_dict(orient='records'):
    summary = '\n'.join([
        f"### {record['task_id']}",
        f"- prompt: `{record['prompt']}`",
        f"- scene: `{record['scene']}`",
        f"- note: `{record['manual_note']}`",
    ])
    display(Markdown(summary))
    display(Video(record['video_path'], embed=True))
